# 1.请求的预处理

In [16]:
from openpyxl.styles.builtins import output
from sympy.physics.units import temperature
from transformers import AutoTokenizer

# 初始化 Tokenizer
tokenizer = AutoTokenizer.from_pretrained("/Users/wjx/Documents/wjx/minimind/minimind-3")
# 示例输入
system_message = {"role": "system", "content": "你是一个历史老师，当学生询问你问题的时候，你总是中文进行回答。"}
user_input = "请问中国的四大发明是什么？"
message = [
    system_message,
    {"role": "user", "content": user_input}
]
# 分词+编码
input_ids = tokenizer.apply_chat_template(message, add_generation_prompt=True, return_tensors="pt")
# 转换消息为模型输入格式
print(f'输入：{user_input}\n编码：{input_ids}')

输入：请问中国的四大发明是什么？
编码：tensor([[   1,  118, 4849,  234,  441, 1001, 1874, 2152, 2025,  294,  899, 2278,
         6090,  441, 5523, 4524,  294,  441, 6056, 3606,  815, 3854,  302,    2,
          234,    1,  832,  311,  234, 5736, 1405,  296, 2405,  565,  532,  851,
         3013,  776,    2,  234,    1, 1388,  570,  811,  234,   25,  234,  234,
           26,  234,  234]])


# 2.推理过程

In [18]:
from transformers import AutoModelForCausalLM
import torch

# 加载模型
model_id = "/Users/wjx/Documents/wjx/minimind/minimind-3"
model = AutoModelForCausalLM.from_pretrained(
    model_id, dtype=torch.bfloat16,  # 量化
    device_map="auto"
)
input_ids = input_ids.to(model.device)
terminators = [tokenizer.eos_token_id,
               tokenizer.convert_tokens_to_ids("<|eot_ids>")]
outputs = model.generate(input_ids,
                         max_new_tokens=256,
                         eos_token_id=terminators,
                         do_sample=True, temperature=0.6,
                         top_p=0.9)
print(f"推理结果：{outputs}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


推理结果：tensor([[   1,  118, 4849,  234,  441, 1001, 1874, 2152, 2025,  294,  899, 2278,
         6090,  441, 5523, 4524,  294,  441, 6056, 3606,  815, 3854,  302,    2,
          234,    1,  832,  311,  234, 5736, 1405,  296, 2405,  565,  532,  851,
         3013,  776,    2,  234,    1, 1388,  570,  811,  234,   25,  234,  234,
           26,  234,  234, 1405,  296, 2405,  565,  532,  851, 3082,  337,  234,
          234,   52,   49,  256, 1394, 3802,  617,  337, 2827, 4703, 4774,  296,
         4169,  130, 3355, 3406,  423, 1394, 3802,  633, 1810,  294, 2483, 3802,
         2370,  296, 2635, 1392, 2255,  294, 1590,  517,  564,  294,  418, 4057,
          296, 2635, 2024,  517, 3449,  581, 4186,  302,  234,  234,   53,   49,
          256, 2357,  318,  151,  617,  337, 1405,  296, 2357,  318,  151,  617,
          532,  851,  294, 4333,  780,  702, 2357,  318,  151,  617,  294, 2483,
         4057,  296, 2357,  318,  151, 1590, 2562,  423, 1487,  565, 5172, 1323,
          294, 2483, 40

# 3.后处理与返回结果

In [26]:
import json
import xml.etree.ElementTree as ET

# 解码和过滤
output_text = tokenizer.decode(outputs[0][input_ids.shape[-1]:93])
print(f"问题:{user_input}\n回复：{output_text}")

# json格式化
json_output = {"问题": user_input, "回复": output_text}
json_formatted = json.dumps(json_output, ensure_ascii=False, indent=4)
print("JSON格式化输出：")
print(json_formatted)

# xml格式化
root = ET.Element("Response")
question = ET.SubElement(root, "问题")
question.text = user_input
answer = ET.SubElement(root, "回复")
answer.text = output_text
xml_formatted = ET.tostring(root, encoding="utf-8").decode("utf-8")
print("XML格式化输出：")
print(xml_formatted)



问题:请问中国的四大发明是什么？
回复：中国的四大发明是指：

1. 造纸术：东汉时期的蔡伦改进了造纸工艺，使得纸张的生产更加高效，质量更高
JSON格式化输出：
{
    "问题": "请问中国的四大发明是什么？",
    "回复": "中国的四大发明是指：\n\n1. 造纸术：东汉时期的蔡伦改进了造纸工艺，使得纸张的生产更加高效，质量更高"
}
XML格式化输出：
<Response><问题>请问中国的四大发明是什么？</问题><回复>中国的四大发明是指：

1. 造纸术：东汉时期的蔡伦改进了造纸工艺，使得纸张的生产更加高效，质量更高</回复></Response>
